# Steam 인디게임 최종 목표 분석 v1

## 목적

이 노트북은 `steam_indie_review_summary.csv`를 제외하고, 전처리된 소스 파일 3개를 기반으로 최종 프로젝트 목표에 맞는 분석을 수행한다.

사용 파일:

- `steam_indie_games.csv`: 게임 단위 메타데이터, 가격, 장르, 태그
- `steam_indie_review_histogram.csv`: 날짜별 추천/비추천 흐름
- `steam_indie_reviews.csv`: 개별 리뷰 본문, 긍정/부정, 플레이타임

분석 목표:

1. 출시 전 관점: 어떤 장르, 태그, 플레이 방식, 가격대가 초기 반응에 유리한가
2. 출시 후 관점: 초기 반응이 나쁜 게임은 무엇이 문제인가
3. 최종 산출물 관점: 대시보드/리포트에 바로 넣을 수 있는 게임 단위 지표와 유형을 만든다

> 핵심 기준은 D7/D30 초기 반응이다. 요약 파일을 쓰지 않고 개별 리뷰의 작성일 기준으로 직접 계산한다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, ast, re, math
from collections import Counter
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_DIR = Path('/mnt/data')
OUTPUT_DIR = DATA_DIR / 'final_analysis_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

GAMES_PATH = DATA_DIR / 'steam_indie_games.csv'
HIST_PATH = DATA_DIR / 'steam_indie_review_histogram.csv'
REVIEWS_PATH = DATA_DIR / 'steam_indie_reviews.csv'

print('데이터 경로 확인')
print(GAMES_PATH.exists(), GAMES_PATH.name)
print(HIST_PATH.exists(), HIST_PATH.name)
print(REVIEWS_PATH.exists(), REVIEWS_PATH.name)

# 1. 공통 함수 정의

전처리 데이터라고 하더라도 분석용 파생 지표는 새로 만든다.

특히 아래 항목은 최종 분석 기준을 통일하기 위해 필요하다.

- 장르/태그 파싱
- 가격대 구간화
- SteamSpy owners 구간화
- Wilson score 계산
- 카테고리에서 플레이 방식 플래그 생성

In [ ]:
def safe_literal_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        val = ast.literal_eval(str(x))
        return val if isinstance(val, list) else []
    except Exception:
        return []


def safe_json_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    try:
        val = json.loads(str(x))
        return val if isinstance(val, dict) else {}
    except Exception:
        try:
            val = ast.literal_eval(str(x))
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}


def wilson_lower_bound(pos, n, z=1.96):
    if n <= 0:
        return np.nan
    phat = pos / n
    denom = 1 + z**2 / n
    center = phat + z**2 / (2*n)
    margin = z * math.sqrt((phat * (1-phat) + z**2/(4*n)) / n)
    return (center - margin) / denom


def weighted_mean(values, weights):
    values = pd.Series(values).astype(float)
    weights = pd.Series(weights).astype(float)
    mask = values.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])


def make_price_bin(price):
    if pd.isna(price):
        return 'unknown'
    if price == 0:
        return 'free'
    if price < 5:
        return 'under_$5'
    if price < 10:
        return '$5-10'
    if price < 20:
        return '$10-20'
    if price < 30:
        return '$20-30'
    return '$30+'


def make_owner_bin(lower, higher):
    if pd.isna(higher):
        return 'unknown'
    if higher <= 20000:
        return '0-20k'
    if higher <= 50000:
        return '20k-50k'
    if higher <= 100000:
        return '50k-100k'
    if higher <= 200000:
        return '100k-200k'
    if higher <= 500000:
        return '200k-500k'
    return '500k+'


def classify_categories(cat_text):
    text = '' if pd.isna(cat_text) else str(cat_text).lower()
    return {
        'has_singleplayer': 'single-player' in text,
        'has_multiplayer': ('multi-player' in text) or ('multiplayer' in text),
        'has_coop': ('co-op' in text) or ('coop' in text),
        'has_online_coop': 'online co-op' in text,
        'has_pvp': 'pvp' in text,
        'has_full_controller': 'full controller support' in text,
        'has_achievements': 'steam achievements' in text,
        'has_cloud': 'steam cloud' in text,
    }


def pct(x):
    return round(float(x) * 100, 2) if pd.notna(x) else np.nan


def metric_by_group(df, group_col, min_games=5):
    rows = []
    for key, g in df.groupby(group_col, dropna=False):
        n_games = g['appid'].nunique()
        if n_games < min_games:
            continue
        rows.append({
            group_col: key,
            'n_games': n_games,
            'median_d30_reviews': g['d30_reviews'].median(),
            'mean_d30_reviews': g['d30_reviews'].mean(),
            'weighted_d30_positive_ratio': weighted_mean(g['d30_positive_ratio'], g['d30_reviews']),
            'median_d30_wilson': g['d30_wilson'].median(),
            'median_late_growth_ratio': g['late_growth_ratio'].median(),
            'median_price': g['price'].median(),
            'median_owners_higher': g['owners_higher'].median(),
        })
    return pd.DataFrame(rows).sort_values(['median_d30_wilson', 'median_d30_reviews'], ascending=False)

# 2. 데이터 로드 및 기본 구조 확인

In [ ]:
games = pd.read_csv(GAMES_PATH)
hist = pd.read_csv(HIST_PATH)
reviews = pd.read_csv(REVIEWS_PATH)

overview = pd.DataFrame([
    {'dataset':'steam_indie_games.csv', 'rows':len(games), 'unique_appid':games['appid'].nunique(), 'main_unit':'game'},
    {'dataset':'steam_indie_review_histogram.csv', 'rows':len(hist), 'unique_appid':hist['appid'].nunique(), 'main_unit':'appid-date'},
    {'dataset':'steam_indie_reviews.csv', 'rows':len(reviews), 'unique_appid':reviews['appid'].nunique(), 'main_unit':'review'},
])

display(overview)
overview.to_csv(OUTPUT_DIR / 'dataset_overview.csv', index=False, encoding='utf-8-sig')

In [ ]:
missing_summary = []
for name, df in [('games', games), ('hist', hist), ('reviews', reviews)]:
    miss = df.isna().sum().sort_values(ascending=False).head(15).reset_index()
    miss.columns = ['column', 'missing_count']
    miss['dataset'] = name
    miss['missing_rate'] = miss['missing_count'] / len(df)
    missing_summary.append(miss[['dataset', 'column', 'missing_count', 'missing_rate']])

missing_summary = pd.concat(missing_summary, ignore_index=True)
display(missing_summary)
missing_summary.to_csv(OUTPUT_DIR / 'missing_summary.csv', index=False, encoding='utf-8-sig')

# 3. 분석용 파생 컬럼 생성

여기서는 원본 전처리 파일을 수정하지 않고, 분석에 필요한 컬럼만 메모리에서 추가한다.

In [ ]:
games['release_date'] = pd.to_datetime(games['release_date'], errors='coerce')
games['year'] = games['release_date'].dt.year
games['release_month'] = games['release_date'].dt.to_period('M').astype(str)
games['genres_list'] = games['genres'].apply(safe_literal_list)
games['tags_dict'] = games['tags'].apply(safe_json_dict)
games['price_bin'] = games['price'].apply(make_price_bin)
games['owners_bin'] = games.apply(lambda r: make_owner_bin(r.get('owners_lower'), r.get('owners_higher')), axis=1)
games['positive_ratio_total'] = np.where(games['total_reviews'] > 0, games['positive'] / games['total_reviews'], np.nan)
games['wilson_total'] = [wilson_lower_bound(p, n) for p, n in zip(games['positive'], games['total_reviews'])]

cat_flags = games['categories'].apply(classify_categories).apply(pd.Series)
games = pd.concat([games, cat_flags], axis=1)

hist['release_date'] = pd.to_datetime(hist['release_date'], errors='coerce')
hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
hist['days_from_release'] = (hist['date'] - hist['release_date']).dt.days
hist['recommendations_total'] = hist['recommendations_up'] + hist['recommendations_down']
hist['positive_ratio'] = np.where(hist['recommendations_total'] > 0, hist['recommendations_up'] / hist['recommendations_total'], np.nan)

reviews['created_date'] = pd.to_datetime(reviews['created_date'], errors='coerce')
reviews = reviews.merge(
    games[['appid','name','release_date','price','price_bin','owners_lower','owners_higher','owners_bin','genres_list','categories',
           'has_singleplayer','has_multiplayer','has_coop','has_online_coop','has_full_controller','has_achievements']],
    on='appid', how='left', suffixes=('', '_game')
)
reviews['days_from_release'] = (reviews['created_date'].dt.normalize() - reviews['release_date']).dt.days
reviews['is_post_release'] = reviews['days_from_release'] >= 0
reviews['is_pre_release'] = reviews['days_from_release'] < 0
reviews['review_len'] = reviews['review'].fillna('').astype(str).str.len()
reviews['review_word_count'] = reviews['review'].fillna('').astype(str).str.split().str.len()

print('파생 컬럼 생성 완료')
print('games:', games.shape)
print('hist:', hist.shape)
print('reviews:', reviews.shape)

# 4. 게임 단위 D7/D30 초기 반응 지표 생성

최종 프로젝트의 핵심은 “초기 반응”이므로, 개별 리뷰 작성일을 출시일과 비교해 기간별 지표를 만든다.

생성 지표:

- `d0_7_reviews`: 출시 후 7일 리뷰 수
- `d0_30_reviews`: 출시 후 30일 리뷰 수
- `d0_30_positive_ratio`: 출시 후 30일 긍정률
- `d0_30_wilson`: 리뷰 수가 적을 때 긍정률 과대평가를 줄이기 위한 보정 긍정 지표
- `d31_90_reviews`: 출시 31~90일 리뷰 수
- `late_growth_ratio`: D30 이후 리뷰 증가 비율

In [ ]:
windows = {
    'pre': lambda d: d < 0,
    'd0_7': lambda d: (d >= 0) & (d <= 7),
    'd0_30': lambda d: (d >= 0) & (d <= 30),
    'd31_90': lambda d: (d >= 31) & (d <= 90),
    'd91_plus': lambda d: d >= 91,
}

base_apps = games[['appid','name','release_date','price','price_bin','owners_lower','owners_higher','owners_bin','genres_list','tags_dict','categories',
                   'has_singleplayer','has_multiplayer','has_coop','has_online_coop','has_full_controller','has_achievements',
                   'positive','negative','total_reviews','positive_ratio_total','wilson_total']].copy()

metric_rows = []
for appid, g in reviews.groupby('appid'):
    row = {'appid': appid}
    for wname, cond in windows.items():
        m = cond(g['days_from_release']) & g['days_from_release'].notna()
        sub = g.loc[m]
        n = len(sub)
        pos = int(sub['voted_up'].sum()) if n else 0
        neg = int((~sub['voted_up']).sum()) if n else 0
        row[f'{wname}_reviews'] = n
        row[f'{wname}_positive'] = pos
        row[f'{wname}_negative'] = neg
        row[f'{wname}_positive_ratio'] = pos / n if n else np.nan
        row[f'{wname}_wilson'] = wilson_lower_bound(pos, n)
        row[f'{wname}_median_playtime_hours'] = sub['playtime_at_review_hours'].median() if n else np.nan
        row[f'{wname}_steam_purchase_rate'] = sub['steam_purchase'].mean() if n else np.nan
        row[f'{wname}_free_received_rate'] = sub['received_for_free'].mean() if n else np.nan
        row[f'{wname}_early_access_rate'] = sub['written_during_early_access'].mean() if n else np.nan
    metric_rows.append(row)

early_metrics = pd.DataFrame(metric_rows)
game_metrics = base_apps.merge(early_metrics, on='appid', how='left')

count_cols = [c for c in game_metrics.columns if c.endswith('_reviews') or c.endswith('_positive') or c.endswith('_negative')]
game_metrics[count_cols] = game_metrics[count_cols].fillna(0)

for wname in windows.keys():
    zero_mask = game_metrics[f'{wname}_reviews'] == 0
    game_metrics.loc[zero_mask, f'{wname}_positive_ratio'] = np.nan
    game_metrics.loc[zero_mask, f'{wname}_wilson'] = np.nan

game_metrics['d30_reviews'] = game_metrics['d0_30_reviews']
game_metrics['d30_positive_ratio'] = game_metrics['d0_30_positive_ratio']
game_metrics['d30_wilson'] = game_metrics['d0_30_wilson']
game_metrics['d7_reviews'] = game_metrics['d0_7_reviews']
game_metrics['late_31_90_reviews'] = game_metrics['d31_90_reviews']
game_metrics['late_growth_ratio'] = np.where(game_metrics['d30_reviews'] > 0, game_metrics['late_31_90_reviews'] / game_metrics['d30_reviews'], np.nan)
game_metrics['d30_review_velocity'] = game_metrics['d30_reviews'] / 30

print('D30 리뷰 1개 이상 게임 수:', int((game_metrics['d30_reviews'] > 0).sum()))
display(game_metrics[['appid','name','release_date','price_bin','d7_reviews','d30_reviews','d30_positive_ratio','d30_wilson','late_31_90_reviews','late_growth_ratio']].sort_values('d30_reviews', ascending=False).head(10))

# 5. 초기 반응 유형화

대시보드와 보고서에서 바로 쓰기 위해 게임을 유형화한다.

유형 기준:

- 초기 흥행 안정형: D30 리뷰 수와 Wilson score가 모두 높은 게임
- 화제성 대비 불안형: D30 리뷰 수는 많지만 Wilson score가 낮은 게임
- 저노출 고품질형: D30 리뷰 수는 적지만 Wilson score가 높은 게임
- 느린 입소문형: 초기 리뷰 수는 낮지만 D31~D90 증가율이 높은 게임
- 초기 반응 취약형: D30 리뷰 수와 Wilson score가 모두 낮은 게임

In [ ]:
def classify_response_type(df):
    m = df['d30_reviews'] > 0
    q_review_hi = df.loc[m, 'd30_reviews'].quantile(0.75)
    q_review_lo = df.loc[m, 'd30_reviews'].quantile(0.25)
    q_wilson_hi = df.loc[m & df['d30_wilson'].notna(), 'd30_wilson'].quantile(0.75)
    q_wilson_lo = df.loc[m & df['d30_wilson'].notna(), 'd30_wilson'].quantile(0.25)
    q_growth_hi = df.loc[df['late_growth_ratio'].notna(), 'late_growth_ratio'].quantile(0.75)
    conditions = [
        (df['d30_reviews'] >= q_review_hi) & (df['d30_wilson'] >= q_wilson_hi),
        (df['d30_reviews'] >= q_review_hi) & (df['d30_wilson'] <= q_wilson_lo),
        (df['d30_reviews'] <= q_review_lo) & (df['d30_wilson'] >= q_wilson_hi),
        (df['d30_reviews'] <= q_review_lo) & (df['late_growth_ratio'] >= q_growth_hi),
        (df['d30_reviews'] <= q_review_lo) & (df['d30_wilson'] <= q_wilson_lo),
    ]
    choices = [
        '초기 흥행 안정형',
        '화제성 대비 불안형',
        '저노출 고품질형',
        '느린 입소문형',
        '초기 반응 취약형',
    ]
    return np.select(conditions, choices, default='일반형'), {
        'q_review_hi': q_review_hi,
        'q_review_lo': q_review_lo,
        'q_wilson_hi': q_wilson_hi,
        'q_wilson_lo': q_wilson_lo,
        'q_growth_hi': q_growth_hi,
    }

game_metrics['response_type'], response_thresholds = classify_response_type(game_metrics)

response_type_summary = (
    game_metrics[game_metrics['d30_reviews'] > 0]
    .groupby('response_type')
    .agg(
        n_games=('appid','nunique'),
        median_d30_reviews=('d30_reviews','median'),
        median_d30_wilson=('d30_wilson','median'),
        median_late_growth_ratio=('late_growth_ratio','median'),
        median_price=('price','median')
    )
    .reset_index()
    .sort_values('n_games', ascending=False)
)

display(pd.DataFrame([response_thresholds]))
display(response_type_summary)

game_metrics.to_csv(OUTPUT_DIR / 'game_level_early_metrics.csv', index=False, encoding='utf-8-sig')
response_type_summary.to_csv(OUTPUT_DIR / 'response_type_summary.csv', index=False, encoding='utf-8-sig')

# 6. 출시 전 관점 분석 1: 가격대별 초기 반응

해석 방향:

- 가격대별로 D30 리뷰 수와 보정 긍정 지표가 어떻게 달라지는지 확인한다.
- 단순 평균보다 `median_d30_reviews`, `weighted_d30_positive_ratio`, `median_d30_wilson`을 함께 본다.
- 표본 수가 너무 적은 가격대는 결론이 아니라 참고로만 본다.

In [ ]:
price_analysis = metric_by_group(game_metrics[game_metrics['d30_reviews'] > 0], 'price_bin', min_games=3)
display(price_analysis)
price_analysis.to_csv(OUTPUT_DIR / 'price_bin_analysis.csv', index=False, encoding='utf-8-sig')

plot_df = price_analysis.sort_values('median_d30_reviews', ascending=True)
plt.figure(figsize=(8, 4))
plt.barh(plot_df['price_bin'], plot_df['median_d30_reviews'])
plt.title('Price bin vs median D30 reviews')
plt.xlabel('Median D30 reviews')
plt.ylabel('Price bin')
plt.tight_layout()
plt.show()

# 7. 출시 전 관점 분석 2: 장르별 초기 반응

장르는 복수 장르가 붙어 있으므로 explode 방식으로 분석한다.

주의:

- `Indie`는 대부분의 게임에 붙어 있어 비교력이 낮으므로 제외한다.
- 장르 하나가 흥행을 결정한다고 해석하면 안 된다.
- 장르별 차이는 가격, 태그, 플레이 방식과 함께 봐야 한다.

In [ ]:
exploded = game_metrics.explode('genres_list').rename(columns={'genres_list':'genre'})

genre_rows = []
for genre, g in exploded[exploded['d30_reviews'] > 0].groupby('genre'):
    if pd.isna(genre) or genre == 'Indie' or len(g) < 5:
        continue
    genre_rows.append({
        'genre': genre,
        'n_games': g['appid'].nunique(),
        'median_d30_reviews': g['d30_reviews'].median(),
        'mean_d30_reviews': g['d30_reviews'].mean(),
        'weighted_d30_positive_ratio': weighted_mean(g['d30_positive_ratio'], g['d30_reviews']),
        'median_d30_wilson': g['d30_wilson'].median(),
        'median_late_growth_ratio': g['late_growth_ratio'].median(),
        'median_price': g['price'].median(),
        'median_owners_higher': g['owners_higher'].median(),
    })

genre_analysis = pd.DataFrame(genre_rows).sort_values(['median_d30_wilson','median_d30_reviews'], ascending=False)
display(genre_analysis)
genre_analysis.to_csv(OUTPUT_DIR / 'genre_analysis.csv', index=False, encoding='utf-8-sig')

plot_df = genre_analysis.sort_values('median_d30_reviews', ascending=True)
plt.figure(figsize=(8, 5))
plt.barh(plot_df['genre'], plot_df['median_d30_reviews'])
plt.title('Genre vs median D30 reviews')
plt.xlabel('Median D30 reviews')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

# 8. 출시 전 관점 분석 3: 플레이 방식별 초기 반응

`categories`에서 싱글/멀티/협동/컨트롤러/도전과제 여부를 플래그화해서 비교한다.

In [ ]:
playmode_rows = []
for col in ['has_singleplayer','has_multiplayer','has_coop','has_online_coop','has_full_controller','has_achievements']:
    temp = game_metrics[game_metrics['d30_reviews'] > 0].copy()
    temp[col] = temp[col].fillna(False).astype(bool)
    for val, label in [(True, 'yes'), (False, 'no')]:
        g = temp[temp[col] == val]
        if len(g) == 0:
            continue
        playmode_rows.append({
            'feature': col,
            'value': label,
            'n_games': g['appid'].nunique(),
            'median_d30_reviews': g['d30_reviews'].median(),
            'weighted_d30_positive_ratio': weighted_mean(g['d30_positive_ratio'], g['d30_reviews']),
            'median_d30_wilson': g['d30_wilson'].median(),
            'median_late_growth_ratio': g['late_growth_ratio'].median(),
        })

playmode_analysis = pd.DataFrame(playmode_rows)
display(playmode_analysis.sort_values(['feature','value']))
playmode_analysis.to_csv(OUTPUT_DIR / 'playmode_analysis.csv', index=False, encoding='utf-8-sig')

# 9. 출시 전 관점 분석 4: 태그별 초기 반응

태그는 Steam 유저 인식에 가까운 정보라서, 장르보다 더 세밀한 포지셔닝 분석에 좋다.

활용 방향:

- 특정 장르 안에서 어떤 태그 조합이 초기 반응에 유리한지 탐색
- 유사 게임 비교 기준으로 사용
- 대시보드에서 태그 필터로 제공

In [ ]:
tag_records = []
for _, row in game_metrics.iterrows():
    td = row.get('tags_dict', {})
    if not isinstance(td, dict):
        continue
    for tag, score in td.items():
        tag_records.append({
            'appid': row['appid'],
            'tag': tag,
            'tag_score': score,
            'd30_reviews': row['d30_reviews'],
            'd30_positive_ratio': row['d30_positive_ratio'],
            'd30_wilson': row['d30_wilson'],
            'late_growth_ratio': row['late_growth_ratio'],
            'price': row['price'],
            'owners_higher': row['owners_higher'],
        })

tag_df = pd.DataFrame(tag_records)

tag_rows = []
for tag, g in tag_df[tag_df['d30_reviews'] > 0].groupby('tag'):
    if len(g) < 5:
        continue
    tag_rows.append({
        'tag': tag,
        'n_games': g['appid'].nunique(),
        'median_tag_score': g['tag_score'].median(),
        'median_d30_reviews': g['d30_reviews'].median(),
        'weighted_d30_positive_ratio': weighted_mean(g['d30_positive_ratio'], g['d30_reviews']),
        'median_d30_wilson': g['d30_wilson'].median(),
        'median_late_growth_ratio': g['late_growth_ratio'].median(),
        'median_price': g['price'].median(),
        'median_owners_higher': g['owners_higher'].median(),
    })

tag_analysis = pd.DataFrame(tag_rows).sort_values(['median_d30_wilson','median_d30_reviews'], ascending=False)
display(tag_analysis.head(30))
tag_analysis.to_csv(OUTPUT_DIR / 'tag_analysis.csv', index=False, encoding='utf-8-sig')

# 10. 초기 반응과 이후 반응 관계

확장 질문에 해당한다.

현재는 장기 흥행 예측이라고 강하게 말하기보다, “초기 반응이 이후 리뷰 흐름과 어느 정도 같이 움직이는가”를 보는 정도로 해석한다.

In [ ]:
sample_game_metrics = game_metrics[game_metrics['appid'].isin(reviews['appid'].unique())].copy()

corr_pairs = [
    ('d7_reviews','d30_reviews'),
    ('d30_reviews','late_31_90_reviews'),
    ('d30_reviews','d91_plus_reviews'),
    ('d30_wilson','d31_90_wilson'),
    ('d30_positive_ratio','d31_90_positive_ratio'),
    ('d30_wilson','d91_plus_wilson'),
]

corr_rows = []
for x, y in corr_pairs:
    temp = sample_game_metrics[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(temp) >= 3:
        r, p = spearmanr(temp[x], temp[y])
        corr_rows.append({'x':x, 'y':y, 'n_games':len(temp), 'spearman_r':r, 'p_value':p})
    else:
        corr_rows.append({'x':x, 'y':y, 'n_games':len(temp), 'spearman_r':np.nan, 'p_value':np.nan})

early_late_corr = pd.DataFrame(corr_rows)
display(early_late_corr)
early_late_corr.to_csv(OUTPUT_DIR / 'early_late_correlation.csv', index=False, encoding='utf-8-sig')

plt.figure(figsize=(6, 5))
plot_df = sample_game_metrics[(sample_game_metrics['d30_reviews'] > 0) & (sample_game_metrics['late_31_90_reviews'] >= 0)]
plt.scatter(plot_df['d30_reviews'], plot_df['late_31_90_reviews'], alpha=0.6)
plt.xscale('log')
plt.yscale('symlog')
plt.title('D30 reviews vs D31-D90 reviews')
plt.xlabel('D30 reviews, log scale')
plt.ylabel('D31-D90 reviews, symlog scale')
plt.tight_layout()
plt.show()

# 11. 출시 후 관점 분석: 부정 리뷰 원인 탐색

여기서는 LLM 감성분석 이전 단계로, 간단한 키워드 기반 이슈 탐색을 한다.

주의:

- 이 결과는 최종 원인 분류가 아니라 “어떤 이슈를 LLM/수작업 분류에서 봐야 하는지” 찾는 용도다.
- `unclassified` 비중이 높으면 키워드 사전을 보강하거나 LLM 분류를 적용해야 한다.

In [ ]:
issue_keywords = {
    'bug_crash': ['bug', 'bugs', 'crash', 'crashes', 'crashed', 'glitch', 'broken', 'freeze', 'softlock', 'stuck', '버그', '튕', '크래시', '오류'],
    'optimization_performance': ['fps', 'lag', 'stutter', 'performance', 'optimization', 'optimisation', 'frame', 'low fps', '최적화', '렉', '프레임'],
    'balance_difficulty': ['balance', 'balanced', 'unbalanced', 'difficulty', 'hard', 'easy', 'grind', 'grindy', '난이도', '밸런스', '노가다'],
    'content_volume': ['content', 'short', 'empty', 'boring', 'repetitive', 'lack', 'no replay', '콘텐츠', '분량', '반복', '지루'],
    'controls_ui': ['control', 'controls', 'keyboard', 'mouse', 'controller', 'ui', 'interface', 'menu', 'tutorial', '조작', '컨트롤', '인터페이스', '튜토리얼'],
    'price_value': ['price', 'expensive', 'worth', 'value', 'money', 'refund', 'overpriced', '가격', '비싸', '환불'],
    'multiplayer_server': ['server', 'matchmaking', 'multiplayer', 'online', 'coop', 'co-op', 'connection', 'disconnect', '서버', '멀티', '매칭', '연결'],
    'translation_language': ['translation', 'language', 'localization', 'localisation', 'english', 'korean', '번역', '한글', '언어'],
}

def detect_issues(text):
    t = str(text).lower()
    found = []
    for issue, kws in issue_keywords.items():
        if any(kw in t for kw in kws):
            found.append(issue)
    return found

reviews['issue_list'] = reviews['review'].fillna('').apply(detect_issues)
reviews['playtime_stage'] = pd.cut(
    reviews['playtime_at_review_hours'],
    bins=[-0.001, 1, 3, 10, 30, np.inf],
    labels=['0-1h','1-3h','3-10h','10-30h','30h+']
)
reviews['review_period'] = pd.cut(
    reviews['days_from_release'],
    bins=[-np.inf, -1, 7, 30, 90, np.inf],
    labels=['pre_release','D0-D7','D8-D30','D31-D90','D91+']
)

neg_reviews = reviews[reviews['voted_up'] == False].copy()
issue_records = []
for _, row in neg_reviews.iterrows():
    issues = row['issue_list'] if isinstance(row['issue_list'], list) else []
    if not issues:
        issues = ['unclassified']
    for issue in issues:
        issue_records.append({
            'appid': row['appid'],
            'name': row['name'],
            'issue': issue,
            'language': row['language'],
            'review_period': row['review_period'],
            'playtime_stage': row['playtime_stage'],
            'days_from_release': row['days_from_release'],
        })

issue_df = pd.DataFrame(issue_records)
issue_summary = (
    issue_df.groupby('issue', dropna=False)
    .agg(negative_review_mentions=('issue','size'), n_games=('appid','nunique'))
    .reset_index()
    .sort_values('negative_review_mentions', ascending=False)
)
issue_summary['mention_share'] = issue_summary['negative_review_mentions'] / issue_summary['negative_review_mentions'].sum()

display(issue_summary)
issue_summary.to_csv(OUTPUT_DIR / 'negative_issue_summary_rule_based.csv', index=False, encoding='utf-8-sig')

plot_df = issue_summary[issue_summary['issue'] != 'unclassified'].sort_values('negative_review_mentions', ascending=True)
plt.figure(figsize=(8, 4))
plt.barh(plot_df['issue'], plot_df['negative_review_mentions'])
plt.title('Negative review issue mentions, rule-based')
plt.xlabel('Mentions')
plt.ylabel('Issue')
plt.tight_layout()
plt.show()

# 12. 플레이타임 기준 부정 이슈 확인

같은 부정 리뷰라도 작성 시점 플레이타임에 따라 의미가 다르다.

예시:

- 0~1시간 부정: 튜토리얼, 조작감, 첫인상, 실행/최적화 문제 가능성
- 10시간 이상 부정: 콘텐츠 부족, 밸런스, 반복성 문제 가능성

In [ ]:
issue_by_playtime = (
    issue_df.groupby(['playtime_stage','issue'], observed=False)
    .size()
    .reset_index(name='negative_review_mentions')
    .sort_values(['playtime_stage','negative_review_mentions'], ascending=[True, False])
)

display(issue_by_playtime.head(40))
issue_by_playtime.to_csv(OUTPUT_DIR / 'negative_issue_by_playtime.csv', index=False, encoding='utf-8-sig')

playtime_vote_summary = (
    reviews.groupby(['playtime_stage','voted_up'], observed=False)
    .agg(review_count=('recommendationid','size'))
    .reset_index()
)

display(playtime_vote_summary)
playtime_vote_summary.to_csv(OUTPUT_DIR / 'playtime_stage_vote_summary.csv', index=False, encoding='utf-8-sig')

# 13. 언어별 리뷰 분포 확인

리뷰 텍스트 분석을 할 때 언어가 섞여 있으면 해석이 흔들릴 수 있다.

따라서 LLM 분석 또는 키워드 분석 전에 언어별 분포를 확인한다.

In [ ]:
language_summary = (
    reviews.groupby('language')
    .agg(
        review_count=('recommendationid','size'),
        positive_ratio=('voted_up','mean'),
        median_playtime_hours=('playtime_at_review_hours','median'),
        n_games=('appid','nunique')
    )
    .reset_index()
    .sort_values('review_count', ascending=False)
)

display(language_summary.head(20))
language_summary.to_csv(OUTPUT_DIR / 'language_review_summary.csv', index=False, encoding='utf-8-sig')

# 14. 대시보드/보고서용 대표 테이블 생성

최종 산출물에 바로 넣기 좋은 테이블을 만든다.

- 초기 반응 우수 게임
- 화제성 대비 불안 게임
- 저노출 고품질 게임
- 게임별 주요 부정 이슈 TOP 3

In [ ]:
leader_cols = ['appid','name','release_date','price','price_bin','owners_bin','d30_reviews','d30_positive_ratio','d30_wilson','late_31_90_reviews','late_growth_ratio','response_type']

top_initial = (
    game_metrics[game_metrics['d30_reviews'] > 0][leader_cols]
    .sort_values(['d30_wilson','d30_reviews'], ascending=False)
    .head(30)
)

risk_games = (
    game_metrics[(game_metrics['d30_reviews'] > 0) & (game_metrics['response_type'] == '화제성 대비 불안형')][leader_cols]
    .sort_values(['d30_reviews','d30_wilson'], ascending=[False, True])
    .head(30)
)

low_exposure_good = (
    game_metrics[(game_metrics['d30_reviews'] > 0) & (game_metrics['response_type'] == '저노출 고품질형')][leader_cols]
    .sort_values(['d30_wilson','d30_reviews'], ascending=[False, True])
    .head(30)
)

per_game_issue = (
    issue_df[issue_df['issue'] != 'unclassified']
    .groupby(['appid','name','issue'])
    .size()
    .reset_index(name='mentions')
)
per_game_issue['rank_in_game'] = per_game_issue.groupby('appid')['mentions'].rank(method='first', ascending=False)
top_game_issues = per_game_issue[per_game_issue['rank_in_game'] <= 3].sort_values(['appid','rank_in_game'])

print('초기 반응 우수 게임')
display(top_initial.head(10))
print('화제성 대비 불안 게임')
display(risk_games.head(10))
print('저노출 고품질 게임')
display(low_exposure_good.head(10))
print('게임별 부정 이슈 TOP 3')
display(top_game_issues.head(20))

top_initial.to_csv(OUTPUT_DIR / 'top_initial_response_games.csv', index=False, encoding='utf-8-sig')
risk_games.to_csv(OUTPUT_DIR / 'risk_games_attention_but_low_quality.csv', index=False, encoding='utf-8-sig')
low_exposure_good.to_csv(OUTPUT_DIR / 'low_exposure_high_quality_games.csv', index=False, encoding='utf-8-sig')
top_game_issues.to_csv(OUTPUT_DIR / 'top_negative_issues_by_game.csv', index=False, encoding='utf-8-sig')

# 15. 현재 분석에서 바로 말할 수 있는 결론 초안

아래 문장은 노션/보고서에 그대로 옮기기보다는, 실제 표와 그래프를 확인한 뒤 조금 다듬어 사용한다.

## 결론 방향

1. 초기 반응은 리뷰 수와 긍정률을 분리해서 보면 안 된다. 리뷰 수가 적은 게임은 긍정률이 높아도 과대평가될 수 있으므로, Wilson score를 함께 사용해야 한다.
2. D7 리뷰 수와 D30 리뷰 수는 매우 강하게 연결되므로, 출시 직후 1주일의 유입 확보가 중요하다.
3. D30 리뷰 수는 이후 31~90일 리뷰 규모와도 함께 움직이는 경향이 있다. 다만 이를 장기 흥행 예측으로 단정하지 말고, 초기 관심이 이후 관심 유지와 연결될 가능성 정도로 해석한다.
4. 출시 전 전략 분석에서는 장르 하나보다 가격대, 태그, 플레이 방식 조합을 함께 봐야 한다.
5. 출시 후 운영 분석에서는 부정 리뷰를 조작/UI, 버그/크래시, 콘텐츠 부족, 가격 대비 가치, 밸런스/난이도 등으로 나누어 패치 우선순위를 제시할 수 있다.

## 다음 단계

- 룰 기반 이슈 분류는 탐색용으로 두고, 최종 보고서에는 LLM 기반 리뷰 원인 분류 결과와 연결하는 것이 좋다.
- 대시보드는 `game_level_early_metrics.csv`를 메인 테이블로 사용하면 된다.
- 상세 게임 진단 페이지는 `top_negative_issues_by_game.csv`를 연결하면 된다.

In [ ]:
summary_text = []
summary_text.append('# 분석 실행 요약')
summary_text.append('')
summary_text.append('## 데이터 규모')
for _, r in overview.iterrows():
    summary_text.append(f"- {r['dataset']}: {r['rows']:,}행, appid {r['unique_appid']:,}개")
summary_text.append('')
summary_text.append('## D30 지표 기준')
summary_text.append(f"- D30 리뷰가 1개 이상 있는 게임: {(game_metrics['d30_reviews'] > 0).sum():,}개")
summary_text.append(f"- D30 리뷰 수 상위 25% 기준: {response_thresholds['q_review_hi']:.1f}개 이상")
summary_text.append(f"- D30 Wilson 상위 25% 기준: {response_thresholds['q_wilson_hi']:.3f} 이상")
summary_text.append('')
summary_text.append('## 가격대별 초기 반응 상위')
for _, r in price_analysis.head(5).iterrows():
    summary_text.append(f"- {r['price_bin']}: n={int(r['n_games'])}, median D30 reviews={r['median_d30_reviews']:.1f}, weighted positive={pct(r['weighted_d30_positive_ratio'])}%")
summary_text.append('')
summary_text.append('## 장르별 초기 반응 상위')
for _, r in genre_analysis.head(8).iterrows():
    summary_text.append(f"- {r['genre']}: n={int(r['n_games'])}, median D30 reviews={r['median_d30_reviews']:.1f}, weighted positive={pct(r['weighted_d30_positive_ratio'])}%")
summary_text.append('')
summary_text.append('## 부정 리뷰 이슈 상위')
for _, r in issue_summary.head(8).iterrows():
    summary_text.append(f"- {r['issue']}: {int(r['negative_review_mentions']):,}건, {pct(r['mention_share'])}%")
summary_text.append('')
summary_text.append('## 초기-이후 반응 상관')
for _, r in early_late_corr.iterrows():
    summary_text.append(f"- {r['x']} vs {r['y']}: n={int(r['n_games'])}, Spearman r={r['spearman_r']:.3f}, p={r['p_value']:.4f}")

summary_md = ''.join(summary_text)
print(summary_md)
(OUTPUT_DIR / 'analysis_summary.md').write_text(summary_md, encoding='utf-8')
print('저장 완료:', OUTPUT_DIR)